# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² colorectal cancer survivor dataset using the `mlcroissant` library.

### Dataset Source
The dataset is provided as a Croissant schema at the following URL:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access top-level metadata fields
meta = dataset.metadata

print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

In [ ]:
# List all available record sets and their fields (by @id)
record_sets = getattr(meta, 'recordSet', None) or []
if hasattr(record_sets, 'to_dict'):
    record_sets = [record_sets]  # single record set
elif not isinstance(record_sets, list):
    record_sets = []
print(f"Number of record sets: {len(record_sets)}")

for i, rs in enumerate(record_sets):
    try:
        rs_id = getattr(rs, '@id', None) or getattr(rs, 'id', None) or getattr(rs, 'identifier', None)
        rs_name = getattr(rs, 'name', None)
        print(f"Record Set {i+1} @id: {rs_id}")
        if rs_name:
            print(f"  Name: {rs_name}")
        fields = getattr(rs, 'field', None) or []
        if hasattr(fields, 'to_dict'):
            fields = [fields]
        field_ids = [getattr(f, '@id', None) for f in fields]
        print("  Fields (@id):", field_ids)
    except Exception as e:
        print(f"Error parsing record set: {e}")

# For convenience later, store all record set @id values
record_set_ids = [getattr(rs, '@id', None) for rs in record_sets if getattr(rs, '@id', None)]
if not record_set_ids:
    print("No record sets found in the metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from above.

In [ ]:
# Extract records for each record set
dataframes = {}
if record_set_ids:
    print("Loading data for record sets...")
    for record_set_id in record_set_ids:
        print(f"  Loading record set: {record_set_id}")
        try:
            records = list(dataset.records(record_set=record_set_id))
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"    Loaded {len(records)} records.")
        except Exception as e:
            print(f"    Failed to load record set {record_set_id}: {e}")

    # Preview columns of the first record set
    first_rs_id = record_set_ids[0]
    print(f"Columns in record set {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print("No record sets to extract.")

## 4. Exploratory Data Analysis (EDA)
Apply exploratory data analysis: filter records, normalize a numeric field, and group by a categorical field, referencing entities by their `@id`.

In [ ]:
# Choose record set and fields to process
if record_set_ids:
    record_set_id = record_set_ids[0]  # Assume first record set as the main data
    df = dataframes[record_set_id]
    print(f"Working with record set: {record_set_id}")
    print(f"Available columns (fields, by @id): {list(df.columns)}")

    # Attempt to select a numeric field - look for typical field names
    # For demonstration, try fields containing 'Age', 'Interval', or similar
    numeric_field_id = None
    for col in df.columns:
        if 'age' in col.lower() or 'interval' in col.lower() or 'count' in col.lower() or 'number' in col.lower():
            numeric_field_id = col
            break

    if numeric_field_id is not None:
        print(f"Selected numeric field (@id): {numeric_field_id}")
        # Ensure the field is numeric
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

        threshold = df[numeric_field_id].mean()  # Use mean as dynamic threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nRecords with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())
            / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized values of {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to select a categorical/group field
        group_field_id = None
        for col in df.columns:
            if ('location' in col.lower() or 'sex' in col.lower() or 'status' in col.lower() or 'type' in col.lower()) and col != numeric_field_id:
                group_field_id = col
                break

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_'+numeric_field_id)
            print(f"\nGrouped data by {group_field_id}:")
            display(grouped_df)
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No suitable numeric field found.")
else:
    print("No data for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Use record set, field, and column `@id`s for accurate referencing.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize only if a numeric field is available
if record_set_ids and 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If group_field_id was found, visualize grouped means
    if 'group_field_id' in locals() and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable numeric field for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and analyze the FAIR² dataset on second primary colorectal cancer survivors using the `mlcroissant` library. By referencing all entities via their `@id`, we ensured reproducible and robust data handling. Explore derived groupings and numeric fields for your own clinical or research questions using this template.